# Create pmtiles for geometries

- Read in geoparquet
- Convert to geojson file
- Use tippecanoe to create pmtiles

In [13]:
import geopandas as gpd
import re

gdf = gpd.read_parquet("../geometries/acsc-ga-2015/out/acsc-primary-compartments.parquet")

geometries_run_id="acsc-ga-2015-run-1"

def slugify(text):
    """Convert text to a slug format suitable for IDs"""
    if text is None:
        return "unknown"
    # Convert to lowercase
    text = str(text).lower()
    # Replace spaces and underscores with hyphens
    text = re.sub(r'[\s_]+', '-', text)
    # Remove invalid characters
    text = re.sub(r'[^a-z0-9\-]', '', text)
    # Remove multiple consecutive hyphens
    text = re.sub(r'-+', '-', text)
    # Strip hyphens from start and end
    text = text.strip('-')
    return text if text else "unknown"


# We need to add an ID column using the get_geometry_name_and_id function (from publish_ace_to_api.ipynb)

def get_geometry_name_and_id(row):
    # Get the name for this feature
    feature_name = (row['name'] if 'name' in row.index else f"Feature {idx}" )
    
    # Create unique ID using slugified name and index
    feature_id = f"{geometries_run_id}-{slugify(feature_name)}"

    return feature_name, feature_id

gdf['id'] = gdf.apply(lambda row: get_geometry_name_and_id(row)[1], axis=1)

print(gdf.head())

gdf.to_file("../geometries/acsc-ga-2015/out/acsc-primary-compartments.geojson", driver="GeoJSON")

   ID_Sequent                        name                            Start  \
0          33           Southern Tasmania                  South East Cape   
1          32                   Storm Bay                      Cape Pillar   
2          34  Davey (Southwest Tasmania)                  South West Cape   
3          31            Eastern Tasmania  Cape Sonnerat (Schouten Island)   
4          30                   St Helens      Cape Barren (Barren Island)   

                               End          Division ID_Primary  Shape_Leng  \
0                  South West Cape  Western Tasmania      TAS06    3.849579   
1                  South East Cape  Eastern Tasmania      TAS05   14.453878   
2                      Cape Sorell  Western Tasmania      TAS07   11.333263   
3                      Cape Pillar  Eastern Tasmania      TAS04    6.997630   
4  Cape Sonnerat (Schouten Island)  Eastern Tasmania      TAS03    8.944435   

   Shape_Area                                           

In [14]:
!tippecanoe --force -z 10 --no-simplification-of-shared-nodes --simplification 10 --drop-densest-as-needed -l "data" -o ../geometries/acsc-ga-2015/out/acsc-primary-compartments.pmtiles ../geometries/acsc-ga-2015/out/acsc-primary-compartments.geojson

100 features, 4898099 bytes of geometry and attributes, 11196 bytes of string pool, 40816320 bytes of vertices, 5240 bytes of nodes
  99.9%  10/858/615  
